# Notebook B — Per-object meshes (TripoSG)

Reads the clean object images from Notebook A (`/content/outputs/objects/*.png`) and turns each into a 3D mesh with **TripoSG**, saving to `/content/outputs/meshes/`.

Run TripoSG in its **own runtime** (don't mix with MoGe/SAM) to avoid dependency clashes.
> TripoSG's API moves; this uses the documented pipeline path with a fallback to its inference script. Check the repo README if the import differs.

## B0 — Get Notebook A's outputs

In [ ]:
import os
# Option 1: upload the zip from Notebook A, then unzip:
from google.colab import files; files.upload()  # notebookA_outputs.zip
# !unzip -q -o notebookA_outputs.zip -d /content/outputs
# os.makedirs("/content/outputs/meshes",exist_ok=True)
# print(sorted(os.listdir("/content/outputs/objects")) if os.path.isdir("/content/outputs/objects") else "upload A's outputs first")

Saving notebookA_outputs(6).zip to notebookA_outputs(6).zip
Buffered data was truncated after reaching the output size limit.

In [1]:
!pip install -q -U numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 102.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.6 which is incompatible.


## B1 — Install TripoSG

In [2]:
%%bash
cd /content
git clone -q https://github.com/VAST-AI-Research/TripoSG.git || echo exists
pip install -q trimesh einops omegaconf huggingface_hub
echo "=== requirements.txt ==="
cat /content/TripoSG/requirements.txt
echo "=== installing line by line ==="
while read -r req; do
  [ -z "$req" ] && continue
  case "$req" in \#*) continue;; esac
  pip install -q "$req" || echo "SKIP (failed to build): $req"
done < /content/TripoSG/requirements.txt
echo "=== done — note any SKIP lines ==="

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 741.0/741.0 kB 3.3 MB/s eta 0:00:00
=== requirements.txt ===
diffusers
transformers
einops
huggingface_hub
opencv-python
trimesh
omegaconf
scikit-image
numpy==1.22.3
peft
jaxtyping
typeguard
diso
pymeshlab # Added for face reduction=== installing line by line ===
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 94.5 MB/s eta 0:00:00
SKIP (failed to build): numpy==1.22.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 3.3 MB/s eta 0:00:00
=== done — note any SKIP lines ===


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [ ]:
!pip install -q pymeshlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.5/106.5 MB 7.3 MB/s eta 0:00:00


In [ ]:
import os, subprocess
env = dict(os.environ, PYTHONPATH="/content/TripoSG")
r = subprocess.run(["python","scripts/inference_triposg.py","--help"],
                   cwd="/content/TripoSG", env=env, capture_output=True, text=True)
print(r.stdout); print(r.stderr[-800:])

usage: inference_triposg.py [-h] --image-input IMAGE_INPUT
                            [--output-path OUTPUT_PATH] [--seed SEED]
                            [--num-inference-steps NUM_INFERENCE_STEPS]
                            [--guidance-scale GUIDANCE_SCALE] [--faces FACES]

options:
  -h, --help            show this help message and exit
  --image-input IMAGE_INPUT
  --output-path OUTPUT_PATH
  --seed SEED
  --num-inference-steps NUM_INFERENCE_STEPS
  --guidance-scale GUIDANCE_SCALE
  --faces FACES

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.



## B2 — Load TripoSG and run on each object

In [ ]:
# import sys, glob, torch, trimesh, os
# sys.path.append("/content/TripoSG")
# DEVICE="cuda" if torch.cuda.is_available() else "cpu"
# DTYPE=torch.float16

# pipe=None
# try:
#     from triposg.pipelines.pipeline_triposg import TripoSGPipeline
#     pipe=TripoSGPipeline.from_pretrained("VAST-AI/TripoSG").to(DEVICE,DTYPE)
#     print("loaded TripoSGPipeline")
# except Exception as e:
#     print("pipeline import differs from this version — fallback to script below.\n",e)

# def run_triposg(img_path, out_path):
#     if pipe is not None:
#         out=pipe(image=img_path)          # adjust kwargs per README (num_inference_steps, etc.)
#         mesh=out.meshes[0] if hasattr(out,"meshes") else out[0]
#         mesh.export(out_path)
#     else:
#         # fallback: call the repo's inference script (edit name/args per README)
#         import subprocess
#         subprocess.run(["python","/content/TripoSG/inference_triposg.py",
#                         "--image",img_path,"--output",out_path], check=False)

# for p in sorted(glob.glob("/content/outputs/objects/*.png")):
#     name=os.path.splitext(os.path.basename(p))[0]
#     out=f"/content/outputs/meshes/{name}.glb"
#     print("->",name);
#     try: run_triposg(p,out); print("   saved",out)
#     except Exception as e: print("   FAILED",e)


import os, glob, subprocess
REPO = "/content/TripoSG"
env = dict(os.environ, PYTHONPATH=REPO)
os.makedirs("/content/outputs/meshes", exist_ok=True)
for p in sorted(glob.glob("/content/outputs/objects/*.png")):
    name = os.path.splitext(os.path.basename(p))[0]
    out  = f"/content/outputs/meshes/{name}.glb"
    print("==>", name)
    r = subprocess.run(
        ["python", "scripts/inference_triposg.py",
         "--image-input", p,
         "--output-path", out,
         "--faces", "30000"],            # decimation target; tune as you like
        cwd=REPO, env=env, capture_output=True, text=True)
    ok = (r.returncode == 0) and os.path.exists(out) and os.path.getsize(out) > 0
    if not ok:
        print(r.stdout[-1200:]); print(r.stderr[-1200:]); print("FAILED", name); continue
    print("   saved", out, os.path.getsize(out), "bytes")

==> obj_00
   saved /content/outputs/meshes/obj_00.glb 546164 bytes
==> obj_01
   saved /content/outputs/meshes/obj_01.glb 539928 bytes
==> obj_02
   saved /content/outputs/meshes/obj_02.glb 542288 bytes
==> obj_03
   saved /content/outputs/meshes/obj_03.glb 537704 bytes
==> obj_04
   saved /content/outputs/meshes/obj_04.glb 540788 bytes
==> obj_05
   saved /content/outputs/meshes/obj_05.glb 540632 bytes
==> obj_06
   saved /content/outputs/meshes/obj_06.glb 538376 bytes
==> obj_07
   saved /content/outputs/meshes/obj_07.glb 541100 bytes
==> obj_08
   saved /content/outputs/meshes/obj_08.glb 540536 bytes
==> obj_09
   saved /content/outputs/meshes/obj_09.glb 536696 bytes


## B3 — Preview the generated meshes

In [ ]:
import trimesh, glob
for p in sorted(glob.glob("/content/outputs/meshes/*.glb")):
    try:
        m=trimesh.load(p); print(p, "verts", len(m.vertices) if hasattr(m,'vertices') else '?')
    except Exception as e: print(p,"load err",e)
# package
import shutil; shutil.make_archive("/content/meshes","zip","/content/outputs/meshes")
try:
    from google.colab import files; files.download("/content/meshes.zip")
except Exception: print("download /content/meshes.zip from Files")

/content/outputs/meshes/obj_00.glb verts ?
/content/outputs/meshes/obj_01.glb verts ?
/content/outputs/meshes/obj_02.glb verts ?
/content/outputs/meshes/obj_03.glb verts ?
/content/outputs/meshes/obj_04.glb verts ?
/content/outputs/meshes/obj_05.glb verts ?
/content/outputs/meshes/obj_06.glb verts ?
/content/outputs/meshes/obj_07.glb verts ?
/content/outputs/meshes/obj_08.glb verts ?
/content/outputs/meshes/obj_09.glb verts ?


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Notes
- The exact pipeline class / kwargs and the script name come from the **TripoSG README** — adjust `run_triposg` accordingly.
- TripoSG is geometry-first; color/texture comes later (vertex colors from clusters in C, or a projection bake).
- T4: run one object at a time; if OOM, reduce resolution / steps per the README.